# Preprocess

## import

In [ ]:
import pandas as pd

## Load

In [ ]:
raw_df = pd.read_csv("../data/raw_data.csv", dtype="str")

In [ ]:
raw_df.shape

In [ ]:
raw_df.describe(include="all").transpose()

## Preprocessing

### URL

In [ ]:
df = raw_df["URL"].to_frame()

### ID

In [ ]:
df["ID"] = raw_df["URL"].map(lambda x: x.split("/")[-2])

In [ ]:
df["ID"].describe()

### category

In [ ]:
df["category"] = raw_df["URL"].map(lambda x: x.split("/")[3])

In [ ]:
df["category"].describe()

### value

In [ ]:
df["values"] = raw_df["価格_"].str.extract(r"(\d*)万円").astype("float")

In [ ]:
df["values"].describe()

欠損値の補填

In [ ]:
values = (
    raw_df["価格"]
    .str.extractall(r"(\d*)万円")
    .unstack()
    .droplevel(level=0, axis=1)
    .astype("float")
)

In [ ]:
values.describe()

In [ ]:
df["values"].fillna(values.min(axis=1), inplace=True)

In [ ]:
df["values"].describe()

### site_area

In [ ]:
df["site_area"] = (
    raw_df["土地面積_"]
    .str.extract(r"(\d+\.*\d*)m2")
    .astype("float")
)

In [ ]:
df["site_area"].describe()

欠損値の補填

In [ ]:
site_area = (
    raw_df["土地面積"]
    .str.split("、", expand=True)[0]
    .str.extractall(r"(\d+\.*\d*)m2")
    .unstack()
    .droplevel(level=0, axis=1)
    .astype("float")
)

In [ ]:
site_area.describe()

In [ ]:
df["site_area"].fillna(site_area.min(axis=1), inplace=True)

In [ ]:
df["site_area"].describe()

### total_floor_area

In [ ]:
df["total_floor_area"] = raw_df["建物面積_"].str.extract(r"(\d+\.*\d*)m2").astype("float")

In [ ]:
df["total_floor_area"].describe()

欠損値の補填

In [ ]:
total_floor_area = (
    raw_df["建物面積"]
    .str.split("、", expand=True)[0]
    .str.extractall(r"(\d+\.*\d*)m2")
    .unstack()
    .droplevel(level=0, axis=1)
    .astype("float")
)

In [ ]:
total_floor_area.describe()

In [ ]:
df["total_floor_area"].fillna(total_floor_area.min(axis=1), inplace=True)

In [ ]:
df["total_floor_area"].describe()

### floor_plan

In [ ]:
df["floor_plan"] = raw_df["間取り_"]

In [ ]:
df["floor_plan"].describe(include="all")

In [ ]:
df["floor_plan"].value_counts(dropna=False)

欠損値の補填

In [ ]:
raw_df["間取り"].value_counts()

In [ ]:
floor_plan = (
    raw_df["間取り"]
    .str.extractall(r"(\dL*D*K\+\d*S|\dL*D*K)")
    .unstack()
    .droplevel(level=0, axis=1)[0]
)

In [ ]:
df["floor_plan"].fillna(floor_plan, inplace=True)

In [ ]:
df["floor_plan"].describe()

## walking_time

In [ ]:
walking_time = (
    raw_df["交通"]
    .str.extract(r"(?P<walking_time>線「[^」]+」歩\d+分)")
)

In [ ]:
df = pd.merge(
    df,
    walking_time["walking_time"].str.extract(r"「(?P<station>.*)」歩(?P<minutes>\d+)分"),
    left_index=True,
    right_index=True
)

## 確認

In [ ]:
df.head()

## 保存

In [ ]:
df.to_csv("../data/data.csv", index=False)